In [ ]:
!pip install -q keras-nightly
!pip install slideio
!pip install keras-cv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 35.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageEnhance, ImageFilter
import slideio
import skimage
from skimage.color import rgb2hed, hed2rgb

# import geopandas as gpd
# from shapely.geometry import Point, LineString, Polygon

# import json
import os
import gc
import shutil
from tqdm import tqdm
import zipfile

# import tensorflow as tf
import keras
import keras_cv

In [ ]:
os.listdir()

['.config', 'sample_data']

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')
gdrive = "/content/gdrive/My Drive/ColabNotebooks/ColonProject/"
os.listdir(gdrive)

Mounted at /content/gdrive/


['Colon_Annotated_Images',
 'CNN_Classifier',
 'Old',
 'Classifier_Tiles',
 'Analyzed_CNN_BCE224',
 'Analyze_SVS_with_CNN_BCE_1024to224.ipynb',
 'colon_research_project',
 'ResearchProjectTiles',
 'Research_Notebooks',
 'Test-CatsDogs']

In [ ]:
#Establish directories, collect SVS files
#Determines which SVS files have already been processed - this runs for a long time.
population = 'control/'

svsdir = os.path.join(gdrive, 'colon_research_project/images/control/additional_control_images')
dstdir_metadata = os.path.join(gdrive, 'ResearchProjectTiles/metadata_02/control')
dstdir_tiles = os.path.join(gdrive, 'ResearchProjectTiles/tiles_02/control')
srcdir_tiles = os.path.join(gdrive, 'ResearchProjectTiles/tiles_02/control_intermediate/')

In [ ]:
print(svsdir)
print(dstdir_metadata)
print(dstdir_tiles)
print(srcdir_tiles)

/content/gdrive/My Drive/ColabNotebooks/ColonProject/colon_research_project/images/control/additional_control_images
/content/gdrive/My Drive/ColabNotebooks/ColonProject/ResearchProjectTiles/metadata_02/control
/content/gdrive/My Drive/ColabNotebooks/ColonProject/ResearchProjectTiles/tiles_02/control
/content/gdrive/My Drive/ColabNotebooks/ColonProject/ResearchProjectTiles/tiles_02/control_intermediate/


In [ ]:
# #load model
modelname = 'CNN_Classifier_BCE_Model_1024x1024_224x224_02.keras'
filepath = '/content/gdrive/My Drive/ColabNotebooks/ColonProject/CNN_Classifier/' + modelname
# shutil.copy(filepath, '/content
shutil.copy(filepath, '/content')
print('Copied!')

model = keras.saving.load_model(modelname)

# # Check its architecture
model.summary()

Copied!


Model: "EfficientNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_2 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 7, 7, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ GMP (GlobalMaxPooling2D)        │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,392,677 (230.38 MB)

 Trainable params: 20,027,457 (76.40 MB)

 Non-trainable params: 310,304 (1.18 MB)

 Optimizer params: 40,054,916 (152.80 MB)

In [ ]:
#Completed tiles in tiles\population
files = os.listdir(dstdir_metadata)
processed = []
for file in files:
    fname = file.split('.')[0]
    fname = fname.split('_')[-1]
    if fname not in processed:
        processed.append(fname)

print('Processed:', len(processed), processed)

Processed: 236 ['179337', '179339', '179342', '179345', '179355', '179352', '179349', '179351', '179356', '179357', '179359', '179360', '179350', '179353', '179465', '179466', '179467', '179470', '179471', '179472', '179468', '179469', '179473', '179483', '179487', '179492', '179493', '179500', '179501', '179499', '179518', '179517', '179519', '179520', '179522', '179524', '179550', '179554', '179553', '179552', '179583', '179601', '179602', '179606', '179605', '179608', '179647', '179327', '179328', '179341', '179338', '179329', '179344', '179330', '179340', '179346', '179331', '179354', '179348', '179364', '179463', '179343', '179462', '179464', '179361', '179363', '179480', '179479', '179478', '179362', '179475', '179491', '179474', '179486', '179495', '179497', '179485', '179498', '179521', '179510', '179513', '179512', '179529', '179511', '179530', '179532', '179539', '179533', '179531', '179534', '179546', '179540', '179551', '179561', '179558', '179576', '179559', '179560', '179

In [ ]:
#Create list of intermediate, cross-check w/ processed to get remaining
files = os.listdir(srcdir_tiles)
intermediate_files = []
for file in files:
    fname = file.split('.')[0]
    if fname in processed:
        continue
    if file.endswith('.zip'):
        intermediate_files.append(file)

print('Remaining', len(intermediate_files), intermediate_files)

Remaining 9 ['331938.zip', '331926.zip', '331918.zip', '331923.zip', '331944.zip', '331920.zip', '331910.zip', '331929.zip', '331906.zip']


In [ ]:
def clean_dir(dir):
    for file in os.listdir(dir):
        filepath = os.path.join(dir, file)
        os.remove(filepath)

In [ ]:
#Copy zip, unzip to subdir, read dataframe/csv
#Eval. tiles, write to dataframe
#Copy svs to subdir
#Extract raw tiles, rescale, save
#Write new csv
#Zip tiles, new csv
#Cleanup folders, move Zip file to permanent folder

#Output sizes
ts = 1024
CNNsize = 224

dirs = ['/content/zipfile/', '/content/svs/',
        '/content/metadata/', '/content/svs_tiles/']

for file in intermediate_files:
    print('##################################')
    #Source name
    fname = file.split('.')[0]

    #Create working folders
    for directory in dirs:
        os.makedirs(directory, exist_ok=True)

    #Copy zip, svs to working folders
    filepath = os.path.join(srcdir_tiles + file)
    shutil.copy(filepath, '/content/zipfile/')
    print('ZIP ', file, ' Copied!')

    #Unzip
    with zipfile.ZipFile('/content/zipfile/' + file, 'r') as zf:
        zf.extractall('/content/zipfile/')

    os.remove('/content/zipfile/' + file)
    print('ZIP ', file, ' Unzipped!')

    df = pd.read_csv('/content/zipfile/CNN_Evaluation_' + fname + '.csv')

    tiles = []
    predictions = []

    #####################################################
    #load tiles
    tilenames = df.tilenames.to_list()
    for tile in tilenames:
        img = Image.open('/content/zipfile/' + tile)
        img = np.array(img)
        tiles.append(img)

    tiles = np.array(tiles)

    #predict
    preds = model.predict(tiles)
    predictions = ['A' if pred < 0.5 else 'B' for pred in preds]

    #keep df consistent with smaller SVS files
    df['predictions'] = predictions
    df['values'] = preds

    #####################################################
    #Copy SVS file
    svsfile = fname + '.svs'
    filepath = os.path.join(svsdir, svsfile)
    shutil.copy(filepath, '/content/svs/')
    print('SVS ', svsfile, ' Copied!')

    #Open SVS
    slidefile = os.path.join('/content/svs/', svsfile)
    slide = slideio.open_slide(slidefile,'SVS')
    scene = slide.get_scene(0)

    #Save reference image
    scale = 10
    refimage = 'CNN_ref_'+fname+'.jpg'
    svs_size = scene.size
    svsimage = scene.read_block((0,0,svs_size[0], svs_size[1]), size=(svs_size[0]//scale,0))
    imgwhole = Image.fromarray(np.uint8(svsimage))
    imgwhole.save(os.path.join('/content/metadata/', refimage))
    print('Refimage1 saved')

    #####################################################
    #Open Refimage2 for colorcoding - visual summary of processing results
    imgarray = np.array(imgwhole)

    #Coordinates, classes from df
    coordinates = df['coords(y1, x1, y2, x2)'].to_list()
    classes = df['predictions'].to_list()

    #Color-code original image and save
    print('Colorcoding ', fname)
    for i, coords in enumerate(coordinates):
        coords = eval(coords)
        y1, x1, y2, x2 = coords
        y1, x1, y2, x2 = int(y1/scale), int(x1/scale), int(y2/scale), int(x2/scale)
        cl = predictions[i]

        if cl == 'A': #green
            imgarray[y1:y2, x1:x2, 1] = 255
        elif cl == 'B': #blue
            imgarray[y1:y2, x1:x2, 2] = 0

    #Resize and save
    codedimage = 'CNN_'+fname+'.jpg'
    imgarray = Image.fromarray(np.uint8(imgarray))
    imgarray = imgarray.resize((imgarray.size[0]//4, imgarray.size[1]//4))
    imgarray.save(os.path.join('/content/metadata/', codedimage))
    print('Colored Image saved')

    del imgarray, imgwhole, svsimage

    #####################################################
    #Extract tiles
    print('Extracting tiles')
    tilenames = []
    for i, coords in tqdm(enumerate(coordinates)):
        #skip if Class B
        if classes[i] == 'B':
            continue
        coords = eval(coords)
        y1, x1, y2, x2 = coords
        imgarray = scene.read_block((x1, y1, ts, ts))

        #to PIL image to resize, save
        img = Image.fromarray(np.uint8(imgarray))
        img = img.resize((CNNsize, CNNsize))

        tilename = fname + '_' + str(i) + '.png'
        img.save(os.path.join('/content/svs_tiles/', tilename))
        tilenames.append(tilename)

    print('Tiles extracted')

    #Zip tiles
    zipname = fname + '.zip'
    #Change dirs to avoid subfolders in zip
    os.chdir('/content/svs_tiles/')
    with zipfile.ZipFile(zipname, 'w') as zf:
        for i, tilename in enumerate(tilenames):
            zf.write(tilename)

    os.chdir('/content/')

    #save CSV file
    csvfile = 'CNN_Evaluation_' + fname + '.csv'
    df.to_csv(os.path.join('/content/metadata/', csvfile))
    print('CSV saved')

    #####################################################
    #Move tiles
    shutil.move(os.path.join('/content/svs_tiles/', zipname), dstdir_tiles)
    shutil.move(os.path.join('/content/metadata/', csvfile), dstdir_metadata)
    shutil.move(os.path.join('/content/metadata/', codedimage), dstdir_metadata)
    shutil.move(os.path.join('/content/metadata/', refimage), dstdir_metadata)

    #Clean working folders
    print('Cleaning folders')
    for directory in dirs:
        clean_dir(directory)
        os.rmdir(directory)


##################################
ZIP  331938.zip  Copied!
ZIP  331938.zip  Unzipped!
42/42 ━━━━━━━━━━━━━━━━━━━━ 17s 258ms/step
SVS  331938.svs  Copied!
Refimage1 saved
Colorcoding  331938
Colored Image saved
Extracting tiles


1328it [00:40, 32.39it/s]


Tiles extracted
CSV saved
Cleaning folders
##################################
ZIP  331926.zip  Copied!
ZIP  331926.zip  Unzipped!
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 202ms/step
SVS  331926.svs  Copied!
Refimage1 saved
Colorcoding  331926
Colored Image saved
Extracting tiles


381it [00:10, 35.15it/s]


Tiles extracted
CSV saved
Cleaning folders
##################################
ZIP  331918.zip  Copied!
ZIP  331918.zip  Unzipped!
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 200ms/step
SVS  331918.svs  Copied!
Refimage1 saved
Colorcoding  331918
Colored Image saved
Extracting tiles


...

Tiles extracted
CSV saved
Cleaning folders
##################################
ZIP  331929.zip  Copied!
ZIP  331929.zip  Unzipped!
67/67 ━━━━━━━━━━━━━━━━━━━━ 9s 138ms/step
SVS  331929.svs  Copied!
Refimage1 saved
Colorcoding  331929
Colored Image saved
Extracting tiles


2115it [01:38, 21.57it/s]


Tiles extracted
CSV saved
Cleaning folders
##################################
ZIP  331906.zip  Copied!
ZIP  331906.zip  Unzipped!
27/27 ━━━━━━━━━━━━━━━━━━━━ 4s 160ms/step
SVS  331906.svs  Copied!
Refimage1 saved
Colorcoding  331906
Colored Image saved
Extracting tiles


851it [00:27, 30.86it/s]


Tiles extracted
CSV saved
Cleaning folders


In [ ]:
#End Runtime
#This works better because shutil in Python is used to copy
#Files are copied back at each iteration in the for-loop above
#If this ran prematurely, only the last iteration files would not be copied.
from google.colab import runtime
runtime.unassign()